In [1]:
import pandas as pd
from pathlib import Path

airbnb = pd.read_csv("../../filtered_listings_reviews-am.csv", low_memory=False)
airbnb.head()

,listing_id,review_id,listing_name,description,review,rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,neighborhood,price,availability_30,availability_60,availability_90,availability_365,availability_eoy
0,27886,2359368,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,"What can I say, Flip provided the perfect Amst...",4.92,4.9,4.94,4.95,4.93,4.9,4.78,Centrum-West,132.0,2,5,16,17,17
1,27886,7509410,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,Just a superb way to spend time in Amsterdam. ...,4.92,4.9,4.94,4.95,4.93,4.9,4.78,Centrum-West,132.0,2,5,16,17,17
2,27886,21170224,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,Why stay anywhere else in the city of canals w...,4.92,4.9,4.94,4.95,4.93,4.9,4.78,Centrum-West,132.0,2,5,16,17,17
3,27886,21700505,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,"The place was beautiful and bright, even when ...",4.92,4.9,4.94,4.95,4.93,4.9,4.78,Centrum-West,132.0,2,5,16,17,17
4,27886,21848508,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,Flip's houseboat is very cool. It is tastefull...,4.92,4.9,4.94,4.95,4.93,4.9,4.78,Centrum-West,132.0,2,5,16,17,17


In [3]:
import numpy as np

def load_glove(path, dim):
    vectors = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            word = parts[0]
            vals = np.asarray(parts[1:], dtype=np.float32)
            if vals.shape[0] != dim:
                continue
            vectors[word] = vals
    return vectors

glove = load_glove("../../glove.6B/glove.6B.100d.txt", dim=100)


In [4]:
import re

token_re = re.compile(r"[a-z]+(?:'[a-z]+)?")

def text_to_glove_avg(text, glove, dim):
    
    tokens = token_re.findall(text.lower())
    vecs = [glove[t] for t in tokens if t in glove]

    if not vecs:
        return np.zeros(dim, dtype=np.float32)

    return np.mean(vecs, axis=0).astype(np.float32)


In [5]:
DIM = 100

airbnb["desc_emb"] = airbnb["description"].apply(lambda x: text_to_glove_avg(x, glove, DIM))
airbnb["review_emb"] = airbnb["review"].apply(lambda x: text_to_glove_avg(x, glove, DIM))
airbnb.head()


,listing_id,review_id,listing_name,description,review,rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,...,review_scores_value,neighborhood,price,availability_30,availability_60,availability_90,availability_365,availability_eoy,desc_emb,review_emb
0,27886,2359368,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,"What can I say, Flip provided the perfect Amst...",4.92,4.9,4.94,4.95,4.93,...,4.78,Centrum-West,132.0,2,5,16,17,17,"[-0.14350858, 0.098111175, 0.2585198, -0.02229...","[-0.152099, 0.20035659, 0.3345409, -0.19615726..."
1,27886,7509410,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,Just a superb way to spend time in Amsterdam. ...,4.92,4.9,4.94,4.95,4.93,...,4.78,Centrum-West,132.0,2,5,16,17,17,"[-0.14350858, 0.098111175, 0.2585198, -0.02229...","[-0.2416597, 0.1784736, 0.28843853, -0.1629926..."
2,27886,21170224,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,Why stay anywhere else in the city of canals w...,4.92,4.9,4.94,4.95,4.93,...,4.78,Centrum-West,132.0,2,5,16,17,17,"[-0.14350858, 0.098111175, 0.2585198, -0.02229...","[-0.11618889, 0.13198675, 0.37054038, -0.15656..."
3,27886,21700505,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,"The place was beautiful and bright, even when ...",4.92,4.9,4.94,4.95,4.93,...,4.78,Centrum-West,132.0,2,5,16,17,17,"[-0.14350858, 0.098111175, 0.2585198, -0.02229...","[-0.16862415, 0.11041213, 0.27169016, -0.18818..."
4,27886,21848508,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,Flip's houseboat is very cool. It is tastefull...,4.92,4.9,4.94,4.95,4.93,...,4.78,Centrum-West,132.0,2,5,16,17,17,"[-0.14350858, 0.098111175, 0.2585198, -0.02229...","[-0.18402936, 0.21676555, 0.30843538, -0.11878..."


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Fit TF-IDF on BOTH fields so weights are meaningful
corpus = pd.concat([airbnb["description"].fillna(""), airbnb["review"].fillna("")], axis=0).astype(str)

tfidf = TfidfVectorizer(lowercase=True, token_pattern=r"[a-z]+(?:'[a-z]+)?", min_df=2)
tfidf.fit(corpus)

vocab = tfidf.vocabulary_
idf = tfidf.idf_

def text_to_glove_tfidf(text, glove, dim, vocab, idf):

    tokens = token_re.findall(text.lower())
    num = np.zeros(dim, dtype=np.float32)
    den = 0.0

    for t in tokens:
        if t in glove and t in vocab:
            w = float(idf[vocab[t]])  # IDF weight
            num += w * glove[t]
            den += w

    if den == 0.0:
        return np.zeros(dim, dtype=np.float32)
    return (num / den).astype(np.float32)


In [7]:
DIM = 100

airbnb["desc_emb_tfidf"] = airbnb["description"].apply(lambda x: text_to_glove_tfidf(x, glove, DIM, vocab, idf))
airbnb["review_emb_tfidf"] = airbnb["review"].apply(lambda x: text_to_glove_tfidf(x, glove, DIM, vocab, idf))
airbnb.head()

,listing_id,review_id,listing_name,description,review,rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,...,price,availability_30,availability_60,availability_90,availability_365,availability_eoy,desc_emb,review_emb,desc_emb_tfidf,review_emb_tfidf
0,27886,2359368,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,"What can I say, Flip provided the perfect Amst...",4.92,4.9,4.94,4.95,4.93,...,132.0,2,5,16,17,17,"[-0.14350858, 0.098111175, 0.2585198, -0.02229...","[-0.152099, 0.20035659, 0.3345409, -0.19615726...","[-0.13715012, 0.06207082, 0.21121688, 0.036976...","[-0.17418899, 0.20681615, 0.27161482, -0.13268..."
1,27886,7509410,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,Just a superb way to spend time in Amsterdam. ...,4.92,4.9,4.94,4.95,4.93,...,132.0,2,5,16,17,17,"[-0.14350858, 0.098111175, 0.2585198, -0.02229...","[-0.2416597, 0.1784736, 0.28843853, -0.1629926...","[-0.13715012, 0.06207082, 0.21121688, 0.036976...","[-0.25615478, 0.19379847, 0.24709517, -0.11718..."
2,27886,21170224,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,Why stay anywhere else in the city of canals w...,4.92,4.9,4.94,4.95,4.93,...,132.0,2,5,16,17,17,"[-0.14350858, 0.098111175, 0.2585198, -0.02229...","[-0.11618889, 0.13198675, 0.37054038, -0.15656...","[-0.13715012, 0.06207082, 0.21121688, 0.036976...","[-0.08221122, 0.21690494, 0.33366668, -0.13936..."
3,27886,21700505,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,"The place was beautiful and bright, even when ...",4.92,4.9,4.94,4.95,4.93,...,132.0,2,5,16,17,17,"[-0.14350858, 0.098111175, 0.2585198, -0.02229...","[-0.16862415, 0.11041213, 0.27169016, -0.18818...","[-0.13715012, 0.06207082, 0.21121688, 0.036976...","[-0.25331497, 0.16720848, 0.23232892, -0.12528..."
4,27886,21848508,"Romantic, stylish B&B houseboat in canal district",Stylish and romantic houseboat on fantastic hi...,Flip's houseboat is very cool. It is tastefull...,4.92,4.9,4.94,4.95,4.93,...,132.0,2,5,16,17,17,"[-0.14350858, 0.098111175, 0.2585198, -0.02229...","[-0.18402936, 0.21676555, 0.30843538, -0.11878...","[-0.13715012, 0.06207082, 0.21121688, 0.036976...","[-0.19037078, 0.23194227, 0.24089305, -0.03601..."


In [ ]:
airbnb.to_csv("airbnb_glove_embeddings-am.csv", index=False)